# ⚡ Módulo 14 - Notebook 03: Partitioning, Caching y Data Skew

## 🗂️ Gestión Avanzada de Particiones y Memoria

**Libro:** Saliendo de lo Pandito  
**Módulo:** 14 - PySpark Optimización ETL Pipelines  
**Duración estimada:** 75 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Gestionar** particiones con repartition/coalesce  
✅ **Aplicar** cache y persist estratégicamente  
✅ **Detectar** data skew  
✅ **Resolver** skew con salting  
✅ **Optimizar** uso de memoria

---

## 📋 Pre-requisitos

* ✅ Notebooks 14_01 y 14_02 completados
* ✅ Conocimiento de DataFrames
* ✅ Familiaridad con conceptos de particionamiento

---

## 📚 Contenido

1. Particionamiento: repartition vs coalesce
2. Cache y Persist
3. StorageLevels
4. Data Skew: Detección
5. Data Skew: Soluciones (Salting)
6. Caso Integrador: Pipeline Optimizado

---

## 💡 Por qué importa

**Particiones y memoria = performance:**

* 🗂️ **Particiones:** Paralelismo efectivo
* 💾 **Cache:** Evitar recomputar
* ⚠️ **Skew:** El cuello de botella oculto
* ⚡ **Optimizado:** 10-100x más rápido

**Dominar estos conceptos = ETLs de producción**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark import StorageLevel

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df.count():,}")
    print(f"   🗂️ Particiones iniciales: {df.rdd.getNumPartitions()}")
    
    # Analizar distribución de particiones
    print(f"\n📊 Análisis de distribución de particiones:")
    partition_sizes = df.rdd.glom().map(len).collect()
    
    print(f"   • Total particiones: {len(partition_sizes)}")
    print(f"   • Registros por partición:")
    print(f"     - Min: {min(partition_sizes):,}")
    print(f"     - Max: {max(partition_sizes):,}")
    print(f"     - Promedio: {sum(partition_sizes)/len(partition_sizes):,.0f}")
    print(f"     - Desv. Estándar: {np.std(partition_sizes):,.0f}")
    
    # Detectar skew
    skew_ratio = max(partition_sizes) / (sum(partition_sizes) / len(partition_sizes))
    print(f"\n   • Skew Ratio: {skew_ratio:.2f}x")
    if skew_ratio > 3:
        print(f"     ⚠️ SKEW SEVERO DETECTADO")
    elif skew_ratio > 2:
        print(f"     ⚠️ SKEW MODERADO DETECTADO")
    else:
        print(f"     ✅ Distribución balanceada")
    
    # Analizar distribución por zona (posible fuente de skew)
    print(f"\n📊 Distribución por zona (posible skew key):")
    df.groupBy("zona").count().orderBy(F.desc("count")).show()
    
    print(f"\n🎯 Este notebook demostrará:")
    print(f"   • Reparticionamiento para balancear carga")
    print(f"   • Coalesce para consolidar archivos")
    print(f"   • Cache para reutilización eficiente")
    print(f"   • Salting para resolver skew")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Partitioning: El Fundamento del Paralelismo

### 🗂️ ¿Qué es una Partición?

**Partición:** Bloque de datos procesable independientemente.

**Relación:**
```
1 Partición = 1 Tarea
10 Particiones = 10 Tareas paralelas
```

**Regla de oro:**
* 100-200 MB por partición
* Número de particiones ≈ 2-4x número de cores

---

### 🔄 repartition() - Redistribución Completa

**Sintaxis:**
```python
# Por número
df.repartition(100)  # 100 particiones, shuffle completo

# Por columna (hash partitioning)
df.repartition("zona")  # Datos de misma zona en misma partición

# Por número y columna
df.repartition(50, "zona")  # 50 particiones, por zona
```

**Costo:** SHUFFLE COMPLETO (📉 Alto)

**Cuándo usar:**
* Antes de joins/aggregations pesadas
* Para balancear particiones desbalanceadas
* Para aumentar paralelismo

---

### 🔽 coalesce() - Reducción Sin Shuffle

**Sintaxis:**
```python
df.coalesce(10)  # Reduce a 10 particiones SIN shuffle
```

**Costo:** BAJO (sin shuffle)

**Limitación:** SOLO para REDUCIR particiones

**Cuándo usar:**
* Antes de escribir (consolidar archivos)
* Después de filtros pesados

**Ejemplo:**
```python
# Tienes 200 particiones pero solo 10 MB de datos
df_filtrado = df.filter("ventas > 1000000")  # Queda poco dato
df_filtrado.coalesce(5).write.parquet("/output")  # 5 archivos en lugar de 200
```

---

### 💾 Cache y Persist

#### ¿Cuándo usar cache?

**Escenario:** DataFrame usado múltiples veces.

**Sin cache:**
```python
df_filtrado = df.filter("ventas > 100000")

# Primera vez: Lee y filtra
df_filtrado.count()  # 10 segundos

# Segunda vez: RE-LEE y RE-FILTRA
df_filtrado.groupBy("zona").sum("ventas")  # Otros 10 segundos
```

**Con cache:**
```python
df_filtrado = df.filter("ventas > 100000").cache()

# Primera vez: Lee, filtra y GUARDA EN MEMORIA
df_filtrado.count()  # 10 segundos

# Segunda vez: LEE DE MEMORIA
df_filtrado.groupBy("zona").sum("ventas")  # 1 segundo
```

---

### 💾 StorageLevels

**persist()** permite elegir dónde almacenar.

```python
from pyspark import StorageLevel

# Solo memoria (rápido, pero limitado)
df.persist(StorageLevel.MEMORY_ONLY)

# Memoria + disco (seguro, si no cabe en RAM)
df.persist(StorageLevel.MEMORY_AND_DISK)

# Serializado (ahorra RAM, usa más CPU)
df.persist(StorageLevel.MEMORY_ONLY_SER)

# Replicado (fault tolerance)
df.persist(StorageLevel.MEMORY_AND_DISK_2)  # 2 copias
```

**Comparativa:**

| Storage Level | RAM | Disco | Serializado | Uso |
|---------------|-----|-------|-------------|-----|
| MEMORY_ONLY | ✅ | ❌ | ❌ | Datos pequeños, RAM suficiente |
| MEMORY_AND_DISK | ✅ | ✅ | ❌ | Default seguro |
| MEMORY_ONLY_SER | ✅ | ❌ | ✅ | Ahorrar RAM |
| DISK_ONLY | ❌ | ✅ | ❌ | Datos temporales grandes |

---

### ⚠️ Data Skew: El Asesino Silencioso

**Data Skew:** Distribución desbalanceada de datos en particiones.

**Problema:**
```
Partición 1: 100 registros    → Termina en 1 segundo
Partición 2: 150 registros    → Termina en 1 segundo
Partición 3: 99,750 registros → Termina en 100 segundos

Tiempo total del job: 100 segundos (esperando a partición 3)
```

**Síntomas:**
* 1 o pocas tareas tardan MUCHO más que las demás
* Uso desigual de recursos (algunos executors ociosos)
* Jobs extremadamente lentos

---

### 🔍 Detectar Skew

**Método 1: Contar por clave**
```python
# Si haces groupBy("zona"), analiza distribución:
df.groupBy("zona").count().orderBy(F.desc("count")).show()

# Si una zona tiene 10x más registros que otras → SKEW
```

**Método 2: Tamaño de particiones**
```python
partition_sizes = df.rdd.glom().map(len).collect()
print(f"Min: {min(partition_sizes)}, Max: {max(partition_sizes)}")

# Si max/promedio > 3x → SKEW
```

---

### 🧂 Salting: Solución al Skew

**Concepto:** Agregar "sal" aleatoria a la clave para distribuir.

**Ejemplo:**

**Problema:**
```
Zona "Centro": 90,000 registros  → 1 partición (lenta)
Zona "Norte":   5,000 registros  → 1 partición (rápida)
Zona "Sur":     5,000 registros  → 1 partición (rápida)
```

**Solución con Salting:**
```python
# Agregar sal aleatoria (0-9)
df_salted = df.withColumn(
    "zona_salted",
    F.concat(F.col("zona"), F.lit("_"), (F.rand() * 10).cast("int"))
)

# Ahora "Centro" se divide en:
# Centro_0, Centro_1, ..., Centro_9 (10 particiones)
df_salted.groupBy("zona_salted").agg(F.sum("ventas"))
```

**Resultado:**
```
Centro_0: 9,000 registros
Centro_1: 9,000 registros
...
Centro_9: 9,000 registros
Norte_0:    500 registros
...
```

Carga balanceada ✅

---

### 🎯 Best Practices

**1️⃣ Repartir ANTES de operaciones pesadas**
```python
df.repartition("clave").join(df2, "clave")  # Evita shuffle en join
```

**2️⃣ Coalesce ANTES de escribir**
```python
df.coalesce(10).write.parquet("/output")  # 10 archivos en lugar de 200
```

**3️⃣ Cache cuando uses 2+ veces**
```python
df_cached = df.filter("complejo").cache()
df_cached.count()
df_cached.groupBy(...).sum(...)
df_cached.unpersist()  # Liberar memoria
```

**4️⃣ Detectar skew temprano**
```python
# Antes de groupBy pesado, analiza distribución
df.groupBy("clave").count().describe().show()
```

**5️⃣ Salting para claves desbalanceadas**
```python
# Si una clave domina (>30% del total), aplicar salting
```

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("🗂️ PARTITIONING, CACHING Y DATA SKEW")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • df.repartition(n) - Redistribuir con shuffle")
print("  • df.coalesce(n) - Reducir sin shuffle")
print("  • df.cache() / df.persist(StorageLevel)")
print("  • Detectar data skew")
print("  • Salting para resolver skew")

print("\n📖 Métodos clave:")
print("  - df.repartition(100, 'columna')  # Hash partitioning")
print("  - df.coalesce(10)  # Consolidar particiones")
print("  - df.cache()  # Almacenar en memoria")
print("  - df.persist(StorageLevel.MEMORY_AND_DISK)")
print("  - df.rdd.getNumPartitions()  # Ver particiones")
print("  - df.rdd.glom().map(len).collect()  # Tamaños")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')